# Fine-tune LFM2-700M on Aaron Rohrbacher Q&A → GGUF

End-to-end: train → merge → convert to GGUF → quantize → download.

**Requirements:** Colab with T4 GPU (Runtime → Change runtime type → T4 GPU). Free tier works.

**Time:** ~60-90 minutes for ~440 examples.

**Outputs:** `LFM2-700M-Q8_0-aaron.gguf` (~790 MB) downloaded to your local machine at the end.


## 1. Install dependencies

Unsloth handles LFM2's hybrid conv+attention architecture automatically and cuts VRAM ~50%.

In [ ]:
# Unsloth + deps. Remove %%capture if present so you can watch progress.
!pip install --upgrade pip
!pip install unsloth
# Pinned transformers/trl/peft for reproducibility
!pip install --upgrade 'transformers>=4.56' 'trl>=0.12' 'peft>=0.13' 'datasets>=3.0' 'bitsandbytes>=0.44' 'accelerate>=1.0' 'sentence-transformers>=3.0'

## 2. Upload training data

Upload these five files from `training/` in the repo:
- `dataset_v2.jsonl` (~326 background Q&A — resume-derived)
- `dataset_v3_gap.jsonl` (~60 Q&A from your own answers in `gap_questions.md`: job-search logistics, tech preferences, experience depth, speaking, crisis stories)
- `dataset_v3_projects.jsonl` (~46 Q&A about portfolio projects from `src/info/Info.jsx`: Session, Heard, Fanboy, AppNow, MOVE, Klear, Thinger, ai, Nuel API, the site itself)
- `dataset_v3_resume.jsonl` (~34 verbatim resume sections from `src/components/resume/Resume.jsx`: each EXPERIENCE entry, each SKILL_GROUP, About text, full-resume overviews, meta-anchor examples)
- `dataset_v3_adversarial.jsonl` (~117 adversarial: declines, redirects, fabrication-bait, pronoun follow-ups, no-document-upload, multi-employer distinction, connect/escalation)

In [ ]:
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

## 3. Load LFM2-700M (4-bit) via Unsloth

Unsloth's pre-quantized build loads faster and uses ~50% less VRAM than stock.

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048  # Worst-case example (system prompt + top-5 facts + date + user + assistant) fits here.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/LFM2-700M-unsloth-bnb-4bit',
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=torch.float16,
)

print('Model:', model.config._name_or_path if hasattr(model.config, '_name_or_path') else type(model).__name__)
print('Chat template present:', tokenizer.chat_template is not None)

## 4. Attach LoRA adapters

Unsloth auto-detects LFM2's attention projection modules and targets them.
Convolution blocks aren't LoRA-targetable by default — that's fine for instruction tuning.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                                   # LoRA rank
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'out_proj',
        'in_proj', 'w1', 'w2', 'w3',
    ],
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
model.print_trainable_parameters()

## 5. Load + format dataset

We prepend a system prompt to every example. The model learns to follow it at inference.

**If you change the system prompt here, change it in `src/components/ChatAgent.jsx` too** — keep training and inference identical.

In [ ]:
import json, os
import numpy as np
from datasets import Dataset
from sentence_transformers import SentenceTransformer

SYSTEM_PROMPT = (
    "You are A-A-Bot, a chat assistant on Aaron Rohrbacher's portfolio site. "
    "You know Aaron's professional background, skills, and projects.\n\n"
    "Follow these rules in order, every turn:\n\n"
    "1. READ the facts below before answering. Your answer must be supported by an explicit statement in the facts.\n"
    "2. DO NOT infer, calculate, estimate, or combine facts to produce new claims. If the facts say \"January 2018\" but don't say \"8 years of experience,\" you do NOT compute the difference \u2014 you quote what's there or decline.\n"
    "3. DO NOT fabricate. If a name, date, number, employer, project, or detail is not literally written in the facts, you do not know it. Making up plausible-sounding information is the worst thing you can do.\n"
    "4. If the facts don't answer the question, say: \"I don't have that info \u2014 just say 'connect me' and I'll open a live chat with Aaron!\"\n"
    "5. For off-topic questions (other people, philosophy, politics, current events, weather, math), briefly redirect back to Aaron.\n"
    "6. Never ask the user to upload, provide, or share any document \u2014 you already know Aaron's background.\n"
    "7. Answer briefly \u2014 one or two sentences."
)

# Fixed training-time date. Mirrors ChatAgent.jsx which injects the live date
# at inference. A stable string during training lets the model generalize the
# rule structure without memorizing any specific day.
TRAINING_DATE = "April 19, 2026"

FACT_CHUNKS = [
    "Aaron Rohrbacher is a Senior Software & DevOps Engineer based in Portland, Oregon. He takes a language-agnostic approach \u2014 he chooses the language and framework for the problem, not the other way around, and is comfortable in just about any stack these days. His GitHub is github.com/aaronrohrbacher and his LinkedIn is linkedin.com/in/aaronrohrbacher. As of March 2026 he is actively seeking his next senior or lead engineering role and is available to start immediately.",
    "About Aaron (in his own words): \"I'm a senior software and DevOps engineer with language-agnostic proficiency in programming and DevOps, and deep expertise in fiduciary finance, HR, payroll, and logistics. I deliver creative, effective, and timely solutions across AWS, GCP, and Azure. Most recently at Forbes AAC, I led emergency stabilization and a ground-up enterprise rebuild of an assistive technology platform, including full rewrites of all native apps (iOS, Android, macOS, Windows, Linux). At SPARQ, I drove AI-powered conversational experiences and infrastructure for enterprise clients including a payroll overhaul serving 500k+ employees. AWS Certified Cloud Practitioner and Certified Developer \u2013 Associate. AWS DevOps Engineer \u2013 Professional in progress. Outside of engineering, I play saxophone and am learning instrument repair.\"",
    "Forbes AAC \u2014 Lead Software Development Engineer (December 2024 \u2013 March 2026, Mansfield OH, remote). This was Aaron's most recent role. Responsibilities and accomplishments: (1) Stabilized a critically failing Ruby 2.6 / Ember 3.0 platform (both past end-of-life), mitigating imminent data and platform loss. Built tooling to convert deprecated Ember code into editable JavaScript \u2014 preserving years of Speech Language Pathologist refinements. (2) Negotiated with Heroku and upgraded infrastructure to prevent complete loss of the application, leaving everything operational with satisfied stakeholders. (3) Rescued Android and iOS apps from deprecation-driven crashes, bringing both to current standards and ensuring uninterrupted service for users who depend on them for daily communication. (4) Architected a complete infrastructure rebuild on Next.js \u2014 injecting compiled legacy Ember JS for continuity \u2014 while medical professionals continued refining decade-old features without interruption. (5) Fully rebuilt cross-platform apps in native code: iOS (Swift), Android (Java/Kotlin), macOS (Swift with native navigation), Windows & Linux (Qt on Rust), replacing deprecated Cordova. (6) Redesigned content sync from one-asset-at-a-time to compressed, licensed package delivery \u2014 saving thousands monthly in data transfer costs and enabling better offline use for AAC users.",
    "SPARQ \u2014 Technical Lead & Senior Software Engineer (August 2022 \u2013 February 2025, Atlanta GA, remote). Aaron was consulted to Forbes AAC via SPARQ and then hired directly by Forbes AAC. Responsibilities and accomplishments: (1) Served as lead system architect building microservices and APIs on AWS Lambda + Node.js with third-party services \u2014 achieving significant cloud infrastructure cost savings. (2) Completed phase-one production deployment of a payroll system overhaul serving 500k+ employees for a global logistics leader. (3) Modernized internal API processes for a new payroll vendor using GCP Cloud Run functions (Python, Java, Node.js), resulting in substantial cost savings. (4) Guided offshore development teams on ADO CI/CD best practices with GCP deployment and local environment virtualization. (5) Mentored junior programmers and introduced modern cloud computing concepts to offshore development teams and stakeholder leadership. (6) Led in-house team to create AI-based conversational experiences for internal client websites. (7) Served on security and risk management teams \u2014 penetration testing and risk assessment.",
    "Nuel Cloud Computing LLC \u2014 Proprietor, Systems Architect & Engineering Director (August 2020 \u2013 February 2024, Portland OR). This was Aaron's own company. Responsibilities and accomplishments: (1) Architected and developed a proprietary AWS solution to provision containerized LAMP stacks (PHP) via ECS and EKS for WordPress designers and PHP developers. Built in Node.js, Express, Python, Serverless Framework, AWS Lambda, API Gateway, ECS/EKS, and Cognito. (2) Architected and designed customer dashboard for automated signup, migration from legacy systems, customized stack preferences, user profiles, and PHP/WordPress plugin management. Frontend in React; backend in Node.js, PHP, and Bash. (3) Delivered the fastest WordPress/PHP installation on the market \u2014 integrated backend caching and CloudFront CDN optimized for WordPress/PHP. (4) Provided security and vulnerability assessments, automating security best practices throughout the system with nearly zero downtime.",
    "Nordic Semiconductor \u2014 Software Engineer II (February 2022 \u2013 March 2022, Portland OR). Responsibilities and accomplishments: (1) Developed test automation tools and scripts for AWS in Node.js using React, Jest, Cypress, Linux, and Istanbul. (2) Built Nordic's first \"thingy lab\" \u2014 enabling IoT device experiments on a proprietary dashboard in novel hardware combinations.",
    "Fiduciary Benchmarks \u2014 Junior Software Development Engineer (July 2018 \u2013 August 2021, Lake Oswego OR). Responsibilities and accomplishments: (1) Designed and maintained automated E2E test suite using JavaScript and Cypress. (2) Designed and maintained sophisticated manual regression test suite using Cucumber and BDD best practices. (3) Analyzed, debugged, and communicated issues reported by data operations and service teams. (4) Reliably communicated test results to Development Manager and agile development team. (5) Implemented WalkMe digital adoption platform \u2014 step-by-step in-app guidance ensuring client success. Note: Fiduciary Benchmarks is a benchmarking company, not a bank.",
    "Planet Argon \u2014 Web Development Intern (January 2018 \u2013 February 2018, Portland OR). This was where Aaron's professional career began. Client projects for NIKE, Aloha Foods, and PAC Global \u2014 test suite enhancements, sitemaps, integration, and advanced Spree eCommerce integration including custom roles.",
    "AI & Machine Learning skills: PyTorch, LLM Implementation & Fine-tuning, NLP, Conversational AI, Amazon Lex, Amazon Polly, Amazon Transcribe, Amazon Q. Aaron's AI/ML work is applied and hands-on \u2014 shipping real systems \u2014 not academic research. He is not a PhD-level ML researcher.",
    "Programming languages \u2014 Aaron is language-agnostic: he chooses the language and framework that fit the problem, not the other way around, and is comfortable in just about any stack these days. For reference, the languages he has shipped production work in include JavaScript, TypeScript, Node.js, Python, Ruby, Java, Kotlin, Swift, Rust, Bash, PowerShell, PHP, and SQL. Do not pitch him as a \"Python guy\" or a specialist in any one language \u2014 the list is representative, not exhaustive, and the point is that he picks the tool for the job.",
    "Frameworks & libraries: Next.js, React, Ruby on Rails, Express, Vue.js, Angular, Ember.js, SST (Serverless Stack), Serverless Framework, Tailwind CSS, SCSS/SASS.",
    "AWS services Aaron has used hands-on: Lambda, ECS/EKS, EC2, API Gateway, CloudFormation, VPC, Route53, IAM, Cognito, S3, CloudFront, RDS, DynamoDB, SES/SNS/SQS, CloudWatch, Cost Management, Systems Manager. AWS is his primary cloud.",
    "GCP & Azure services: Cloud Run, Cloud Functions, Compute Engine, Cloud SQL, Pub/Sub, Azure DevOps, plus Azure equivalents of core AWS services. He has shipped production code on all three major clouds.",
    "Infrastructure & DevOps: Docker, Kubernetes, Terraform, AWS CDK, CloudFormation, GitHub Actions, GitLab CI, Azure DevOps, Jenkins, Microservices, Serverless Architecture. He has designed and operated serverless microservices architectures end-to-end.",
    "Mobile & desktop development: iOS (Swift/SwiftUI), Android (Java/Kotlin), macOS (Swift/AppKit), Windows & Linux (Qt/Rust), React Native, Cordova. Most of this experience came from the Forbes AAC native rewrite.",
    "Databases: PostgreSQL, MySQL/MariaDB, MongoDB, DynamoDB, Redis, ElastiCache.",
    "Testing & QA: Cypress, Jest, Jasmine, Mocha, Selenium, Puppeteer, TestNG, Rest-assured, BDD/Cucumber.",
    "Security & compliance: Penetration Testing, Burp Suite, OWASP Top 10, SOC 2 Preparation, IAM & Access Control, Encryption & Data Protection. Aaron served on SPARQ's security and risk management team doing pentesting and risk assessment. He is not a dedicated security engineer.",
    "Frontend & design: HTML5, CSS3, Bootstrap, Responsive Design, Accessibility (WCAG), UI/UX Optimization, Adobe Creative Suite, GIMP.",
    "Additional technologies: WordPress/PHP, VoIP/SIP Trunking, WebRTC, Real-time Communication, CDN & Caching (Varnish, Redis), Linux administration (Ubuntu, CentOS, Debian, Arch), Agile/Scrum.",
    "Certifications: AWS Certified Cloud Practitioner (held), AWS Certified Developer \u2013 Associate (held), AWS Certified DevOps Engineer \u2013 Professional (in progress).",
    "Outside of engineering, Aaron plays saxophone and is learning instrument repair. He also plays clarinet and records music as an amateur audio engineer. He has one brother.",
    "What Aaron is NOT: He has never worked for a government agency, a hospital or healthcare organization, or a bank. Fiduciary Benchmarks (2018\u20132021) is a benchmarking company, not a bank. Forbes AAC was his MOST RECENT employer, not his first \u2014 his career began in January 2018 at Planet Argon. He is not a PhD-level or academic ML researcher; his AI/ML work is applied. Do not invent employers, degrees, certifications, or personal details that are not stated above.",
]

# Mirror of ChatAgent.selectRelevantFacts: top-5 by cosine sim, plus index 0
# pinned as identity anchor. The embedder used at runtime is
# Xenova/all-MiniLM-L6-v2 (quantized port of sentence-transformers/all-MiniLM-L6-v2).
# Loading the HF source model here gives near-identical embeddings.
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
fact_embs = embedder.encode(FACT_CHUNKS, normalize_embeddings=True, convert_to_numpy=True)

def select_relevant_facts(query, top_k=5):
    q = embedder.encode([query], normalize_embeddings=True, convert_to_numpy=True)[0]
    scores = fact_embs @ q
    top_idx = set(np.argsort(-scores)[:top_k].tolist())
    top_idx.add(0)  # identity anchor
    return [FACT_CHUNKS[i] for i in sorted(top_idx)]

def build_system_prompt(user_query):
    facts = "\n\n".join(select_relevant_facts(user_query, top_k=5))
    return f"{SYSTEM_PROMPT}\n\nToday's date is {TRAINING_DATE}.\n\n{facts}"

def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

background = load_jsonl(os.path.join(DATA_DIR, 'dataset_v2.jsonl'))
gap = load_jsonl(os.path.join(DATA_DIR, 'dataset_v3_gap.jsonl'))
projects = load_jsonl(os.path.join(DATA_DIR, 'dataset_v3_projects.jsonl'))
resume = load_jsonl(os.path.join(DATA_DIR, 'dataset_v3_resume.jsonl'))
adversarial = load_jsonl(os.path.join(DATA_DIR, 'dataset_v3_adversarial.jsonl'))
all_records = background + gap + projects + resume + adversarial
print(f'Total examples: {len(all_records)} '
      f'({len(background)} bg + {len(gap)} gap + {len(projects)} projects + '
      f'{len(resume)} resume + {len(adversarial)} adversarial)')

# Build prompt-completion rows. TRL recognizes this shape and auto-applies
# completion_only_loss=True, so gradient flows only through assistant tokens.
# See https://huggingface.co/docs/trl/en/sft_trainer (prompt-completion format).
def build_embed_query(msgs):
    """Mirror ChatAgent.jsx embedQuery: last user + prior exchange if present.
    Runtime (ChatAgent.jsx ~415-422) joins prior user/assistant turns with the
    current user query. We reproduce that here so training RAG matches inference.
    """
    user_turns = [i for i, m in enumerate(msgs) if m['role'] == 'user']
    assert user_turns, 'example has no user turn'
    last_user_idx = user_turns[-1]
    last_user = msgs[last_user_idx]['content']
    # Runtime slices history[-3:-1] — the two turns immediately before current user.
    prior = [m['content'] for m in msgs[max(0, last_user_idx-2):last_user_idx]]
    prior_str = ' '.join(prior).strip()
    return f'{prior_str} {last_user}'.strip() if prior_str else last_user

def to_prompt_completion(ex):
    msgs = ex['messages']
    embed_query = build_embed_query(msgs)
    system = build_system_prompt(embed_query)

    last_assistant_idx = max((i for i, m in enumerate(msgs) if m['role'] == 'assistant'), default=-1)
    assert last_assistant_idx >= 0, f'example has no assistant turn: {ex}'

    prompt_msgs = [{'role': 'system', 'content': system}]
    for i, m in enumerate(msgs):
        if i < last_assistant_idx:
            prompt_msgs.append(m)
    completion_msgs = [msgs[last_assistant_idx]]
    return {'prompt': prompt_msgs, 'completion': completion_msgs}

formatted = [to_prompt_completion(ex) for ex in all_records]
dataset = Dataset.from_list(formatted)
train_dataset = dataset.shuffle(seed=42)
print(f'Training set: {len(train_dataset)} examples (prompt-completion format)')
print('\nSample prompt system turn (first 500 chars):')
print(train_dataset[0]['prompt'][0]['content'][:500])
print('\nSample completion:')
print(train_dataset[0]['completion'][0]['content'][:300])


## 6. Train

3 epochs, batch 2 with grad accum 4 (effective batch size 8), linear LR schedule with 5% warmup.
Expect ~30-60 minutes on T4 for ~880 examples (packing off).

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir='./outputs',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='linear',
    warmup_ratio=0.05,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy='epoch',
    max_length=MAX_SEQ_LENGTH,      # renamed from max_seq_length in newer TRL
    packing=False,
    optim='adamw_8bit',
    report_to='none',
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    args=training_args,
)

trainer.train()

## 7. Quick sanity check

Run a couple of queries before conversion to make sure training worked.

In [ ]:
# Sanity check the trained model before merging. We intentionally do NOT call
# FastLanguageModel.for_inference(model) here — it mutates in-memory state in
# ways that can break the merge step below. Generation is slightly slower
# (~20-30s for six probes instead of ~5s) but merge stays reliable.

def ask(question):
    msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': question},
    ]
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt',
    ).to('cuda')
    out = model.generate(
        inputs, max_new_tokens=120, do_sample=True, temperature=0.7, top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f'Q: {question}\nA: {text}\n')

ask('What does Aaron do?')
ask('What AWS services does he know?')
ask('How many brothers does Aaron have?')
ask('What is the meaning of life?')
ask('What is Aaron\'s wife\'s name?')
ask('Can you read his resume?')

## 8. Merge LoRA into base model

GGUF conversion needs a standalone model, not a LoRA adapter. Unsloth's `save_pretrained_merged` handles dequantization + merge in one step.

In [ ]:
import os, torch

MERGED_DIR = '/content/lfm2-700m-aaron-merged'

# Primary: Unsloth's save_pretrained_merged handles 4-bit → fp16 dequant,
# LoRA merge, and save in one call. Absolute path avoids cwd issues.
# Fallback: manual merge_and_unload with save_original_format=False to
# sidestep the NotImplementedError some transformers versions throw when
# trying to reverse 4-bit weight conversions during save.

def merged_looks_valid(path):
    return (
        os.path.isdir(path)
        and os.path.isfile(os.path.join(path, 'config.json'))
        and any(f.endswith('.safetensors') or f.endswith('.bin')
                for f in os.listdir(path))
    )

primary_err = None
try:
    print('[merge] primary path: Unsloth save_pretrained_merged(merged_16bit)')
    model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method='merged_16bit')
except Exception as e:
    primary_err = e
    print('[merge] primary failed:', type(e).__name__, '-', str(e)[:200])

if not merged_looks_valid(MERGED_DIR):
    print('[merge] falling back to manual merge_and_unload + save_pretrained(save_original_format=False)')
    try:
        merged = model.merge_and_unload()
    except Exception as e:
        print('[merge] merge_and_unload failed:', type(e).__name__, '-', str(e)[:200])
        raise
    try:
        merged.save_pretrained(MERGED_DIR, safe_serialization=True, save_original_format=False)
    except TypeError:
        # Older transformers doesn't accept save_original_format — try without.
        merged.save_pretrained(MERGED_DIR, safe_serialization=True)
    tokenizer.save_pretrained(MERGED_DIR)

if not merged_looks_valid(MERGED_DIR):
    raise RuntimeError(
        f'Both merge paths failed. Primary error: {primary_err}. '
        f'Check {MERGED_DIR} contents.'
    )

print('[merge] OK — merged model at', MERGED_DIR)
!ls -la {MERGED_DIR}

## 9. Clone and build llama.cpp

We need `convert_hf_to_gguf.py` (has native LFM2 support) and `llama-quantize`.

In [ ]:
# Clone + build llama.cpp (just the llama-quantize target). ~2-5 min.
# Progress will scroll as it runs — git clone → pip install → cmake configure → compile.
!git clone --depth=1 https://github.com/ggml-org/llama.cpp.git /content/llama.cpp
!pip install -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
# Build just llama-quantize (not the full project — saves 5+ min)
!cmake -B /content/llama.cpp/build /content/llama.cpp -DGGML_CUDA=OFF -DLLAMA_BUILD_SERVER=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF
!cmake --build /content/llama.cpp/build --target llama-quantize -j $(nproc)

## 10. Convert merged model → GGUF (F16)

In [ ]:
F16_PATH = '/content/lfm2-700m-aaron-f16.gguf'
# Absolute path — robust regardless of current working directory.
!python /content/llama.cpp/convert_hf_to_gguf.py /content/lfm2-700m-aaron-merged --outfile {F16_PATH} --outtype f16
!ls -lh {F16_PATH}

## 11. Quantize F16 → Q8_0

Q8_0 is what the site currently loads (`LFM2-700M-Q8_0.gguf`). Near-FP16 quality, ~50% smaller file.

In [ ]:
Q8_PATH = '/content/LFM2-700M-Q8_0-aaron.gguf'
!/content/llama.cpp/build/bin/llama-quantize {F16_PATH} {Q8_PATH} Q8_0
!ls -lh {Q8_PATH}

## 12. Download the quantized GGUF

This triggers a browser download. The file lands in your Downloads folder.

**Next steps (local):**
1. Move the downloaded file to `public/models/lfm2-700m-gguf/LFM2-700M-Q8_0-aaron.gguf` in the repo
2. Update `src/components/ChatAgent.jsx` to point at the new filename
3. Update the system prompt in `ChatAgent.jsx` to match `SYSTEM_PROMPT` from step 5

In [ ]:
files.download(Q8_PATH)